In [ ]:
import os, json, joblib, pickle, torch, re
import numpy as np
import pandas as pd
import deepchem as dc
import tensorflow as tf
import selfies as sf
from rdkit import Chem
from rdkit.Chem import Draw


# ==============================================================
# ======================== CONFIG ===============================
# ==============================================================
BASE_DIR = r"D:\Final2"
DATA_FILE = os.path.join(BASE_DIR, r"A&NA_cleaned_0.08.pkl")

STAGE1_MODEL = os.path.join(BASE_DIR, "Sub_classification_model", "NA&A", "best_model.h5")
STAGE1_LABEL = os.path.join(BASE_DIR, "Sub_classification_model", "NA&A", "label_encoder.pkl")

STAGE2_A_MODEL = os.path.join(BASE_DIR, "Sub_classification_model", "A", "best_model.h5")
STAGE2_A_LABEL = os.path.join(BASE_DIR, "Sub_classification_model", "A", "label_encoder_cleaned_A.pkl")

STAGE2_NA_MODEL = os.path.join(BASE_DIR, "Sub_classification_model", "NA", "best_model.h5")
STAGE2_NA_LABEL = os.path.join(BASE_DIR, "Sub_classification_model", "NA", "label_encoder_cleaned_NA.pkl")

LAST_STAGE_A_DIR = os.path.join(BASE_DIR, "last_stage_models_A")
LAST_STAGE_NA_DIR = os.path.join(BASE_DIR, "last_stage_models_NA")
CLUSTER_BASE_DIR = os.path.join(BASE_DIR, "Last_stage_model_with_clustering")

STAGE1_OUT = os.path.join(BASE_DIR, "Results_stage_1")
STAGE2_OUT = os.path.join(BASE_DIR, "Results_stage_2")
STAGE3_NOCLUST_OUT = os.path.join(BASE_DIR, "Results_stage_3_NoClustering")
STAGE3_CLUST_OUT = os.path.join(BASE_DIR, "Results_stage_3_Clustering")

os.makedirs(STAGE1_OUT, exist_ok=True)
os.makedirs(STAGE2_OUT, exist_ok=True)
os.makedirs(STAGE3_NOCLUST_OUT, exist_ok=True)
os.makedirs(STAGE3_CLUST_OUT, exist_ok=True)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path


# ==============================================================
# ===================== SAVE HELPERS ============================
# ==============================================================

def safe_filename(s: str, maxlen: int = 120) -> str:
    s = "" if s is None else str(s)
    s = s.strip()
    s = re.sub(r'[<>:"/\\|?*\n\r\t]', "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:maxlen] if len(s) > maxlen else s


def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    mz_cols = [c for c in df.columns if str(c).startswith("mz")]
    front = [c for c in ["mol_id", "Stage1_Pred", "Stage2_Pred", "Cluster_ID",
                         "Pred_SELFIES", "Pred_SMILES", "Structure_File"]
             if c in df.columns]
    other = [c for c in df.columns if c not in front and c not in mz_cols]
    return df[front + other + mz_cols]


def save_df_intermediate(df: pd.DataFrame, save_dir: str, base_name: str,
                         save_pkl=True, save_csv=True, save_json=True,
                         json_orient: str = "records"):
    """
    For Stage 1 & Stage 2 outputs:
    Save same dataframe as PKL + CSV + JSON.
    """
    ensure_dir(save_dir)
    df_out = reorder_columns(df.copy())

    base = os.path.join(save_dir, base_name)
    paths = {}

    if save_pkl:
        pkl_path = base + ".pkl"
        df_out.to_pickle(pkl_path)
        paths["pkl"] = pkl_path

    if save_csv:
        csv_path = base + ".csv"
        df_out.to_csv(csv_path, index=False, encoding="utf-8")
        paths["csv"] = csv_path

    if save_json:
        json_path = base + ".json"
        df_out.to_json(json_path, orient=json_orient, force_ascii=False)
        paths["json"] = json_path

    return paths


def save_predictions_multi(df_original: pd.DataFrame,
                           selfies_list,
                           smiles_list,
                           save_dir: str,
                           group_name: str,
                           json_orient: str = "records"):
    """
    Final Stage 3 output:
      - XLSX + CSV + JSON + PKL
      - structure PNGs named by mol_id and SMILES
    """
    ensure_dir(save_dir)

    df_out = df_original.copy()

    if "mol_id" not in df_out.columns:
        df_out = df_out.reset_index(drop=True)
        df_out["mol_id"] = np.arange(len(df_out)) + 1

    df_out["Pred_SELFIES"] = list(selfies_list)
    df_out["Pred_SMILES"] = list(smiles_list)

    # Save structures
    structures_dir = ensure_dir(os.path.join(save_dir, "structures"))
    structure_files = []

    for _, row in df_out.iterrows():
        mid = int(row["mol_id"])
        smi = row["Pred_SMILES"]
        if smi:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                smi_safe = safe_filename(smi, maxlen=80)
                fname = f"mol_{mid:06d}__{smi_safe}.png"
                Draw.MolToFile(mol, os.path.join(structures_dir, fname))
                structure_files.append(os.path.join("structures", fname))
            else:
                structure_files.append("")
        else:
            structure_files.append("")

    df_out["Structure_File"] = structure_files
    df_out = reorder_columns(df_out)

    base = os.path.join(save_dir, f"{group_name}_final_predictions")

    # XLSX
    xlsx_path = base + ".xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_out.to_excel(writer, index=False, sheet_name="predictions")

    # CSV
    csv_path = base + ".csv"
    df_out.to_csv(csv_path, index=False, encoding="utf-8")

    # JSON
    json_path = base + ".json"
    df_out.to_json(json_path, orient=json_orient, force_ascii=False)

    # PKL
    pkl_path = base + ".pkl"
    df_out.to_pickle(pkl_path)

    # SELFIES TXT (optional)
    selfies_path = os.path.join(save_dir, f"{group_name}_selfies.txt")
    with open(selfies_path, "w", encoding="utf-8") as f:
        for s in selfies_list:
            f.write(str(s) + "\n")

    print(f"✅ Saved XLSX → {xlsx_path}")
    print(f"✅ Saved CSV  → {csv_path}")
    print(f"✅ Saved JSON → {json_path}")
    print(f"✅ Saved PKL  → {pkl_path}")
    print(f"✅ Saved structures → {structures_dir}")

    return {"xlsx": xlsx_path, "csv": csv_path, "json": json_path, "pkl": pkl_path}


def pad_or_truncate(X: np.ndarray, target_features: int) -> np.ndarray:
    if X.shape[1] < target_features:
        X = np.pad(X, ((0, 0), (0, target_features - X.shape[1])))
    return X[:, :target_features]


# ==============================================================
# ================== STAGE 1: A / NA ============================
# ==============================================================

def stage1_classify(data_path, model_path, label_path, output_dir):
    print("\n=== Stage 1: A/NA Classification ===")
    df = pd.read_pickle(data_path)

    # Stable id for each spectrum
    if "mol_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["mol_id"] = np.arange(len(df)) + 1

    mz_cols = [c for c in df.columns if str(c).startswith("mz")]
    if not mz_cols:
        raise ValueError("No mz... columns found in dataset.")

    X = df[mz_cols].values.astype(np.float32).reshape(len(df), -1, 1)

    model = tf.keras.models.load_model(model_path)
    le = joblib.load(label_path)

    preds = model.predict(X, verbose=0)
    y_idx = preds.argmax(axis=1)
    df["Stage1_Pred"] = le.inverse_transform(y_idx)

    stage1_dir = ensure_dir(os.path.join(output_dir, "Results_stage_1"))

    for label in df["Stage1_Pred"].unique():
        sub = df[df.Stage1_Pred == label].copy()
        save_df_intermediate(
            sub, stage1_dir,
            base_name=f"predicted_dataset_{label}",
            save_pkl=True, save_csv=True, save_json=True
        )

    print("✅ Stage 1 completed.")
    return stage1_dir


# ==============================================================
# ================== STAGE 2: SUBGROUPS =========================
# ==============================================================

def stage2_classify(branch, stage1_dir, model_path, label_path):
    print(f"\n=== Stage 2: {branch} Sub-group Classification ===")

    input_path = os.path.join(stage1_dir, f"predicted_dataset_{branch}.pkl")
    if not os.path.exists(input_path):
        print(f"⚠️ No data for {branch}, skipping.")
        return None

    df = pd.read_pickle(input_path)
    mz_cols = [c for c in df.columns if str(c).startswith("mz")]
    if not mz_cols:
        raise ValueError(f"No mz... columns found for Stage2 branch {branch}.")

    X = df[mz_cols].values.astype(np.float32).reshape(len(df), -1, 1)

    model = tf.keras.models.load_model(model_path)
    le = joblib.load(label_path)

    preds = model.predict(X, verbose=0)
    y_idx = preds.argmax(axis=1)
    df["Stage2_Pred"] = le.inverse_transform(y_idx)

    branch_dir = ensure_dir(os.path.join(STAGE2_OUT, branch))

    for lbl in df["Stage2_Pred"].unique():
        sub = df[df.Stage2_Pred == lbl].copy()
        save_df_intermediate(
            sub, branch_dir,
            base_name=f"{lbl}_predictions",
            save_pkl=True, save_csv=True, save_json=True
        )
        print(f"✅ Saved {lbl} → {branch_dir} (PKL/CSV/JSON)")

    return branch_dir


# ==============================================================
# ============== STAGE 3A  (No-Clustering) ======================
# ==============================================================

def load_deepchem_model(model_dir):
    with open(os.path.join(model_dir, "architecture.json")) as f:
        arch = json.load(f)
    model = dc.models.MultitaskClassifier(
        n_tasks=arch["n_tasks"],
        n_features=arch["n_features"],
        layer_sizes=arch["layer_sizes"],
        dropouts=arch["dropout_rate"],
        model_dir=model_dir
    )
    model.restore()
    return model, arch


def load_symbol_dict(model_dir):
    f = os.path.join(model_dir, "selfies.json")
    if not os.path.exists(f):
        for x in os.listdir(model_dir):
            if "selfies" in x and x.endswith(".json"):
                f = os.path.join(model_dir, x)
                break
    with open(f) as j:
        d = json.load(j)
    return {k: int(v) for k, v in d.items()}


def predict_no_clustering(df, model_dir, save_dir, group_name):
    model, arch = load_deepchem_model(model_dir)

    vocab = load_symbol_dict(model_dir)          # token -> idx
    itos = {v: k for k, v in vocab.items()}      # idx -> token
    vocab_size = len(itos)

    mz_cols = [c for c in df.columns if str(c).startswith("mz")]
    X = df[mz_cols].values.astype(np.float32)
    X = pad_or_truncate(X, int(arch["n_features"]))

    dataset = dc.data.NumpyDataset(
        X=X,
        y=np.zeros((len(X), arch["n_tasks"]), dtype=np.float32)
    )

    try:
        preds = model.predict(dataset)
    except Exception:
        with torch.no_grad():
            outputs = model.model(torch.tensor(X, dtype=torch.float32))
            outputs = outputs[0] if isinstance(outputs, tuple) else outputs
            preds = outputs.cpu().numpy()

    selfies_list, smiles_list = [], []

    for pred in preds:
        if pred.ndim == 2 and pred.shape[-1] == 2:
            pr = pred[:, 1]
        else:
            pr = pred

        bits = (pr >= 0.5).astype(int)
        usable = (len(bits) // vocab_size) * vocab_size
        bits = bits[:usable]

        seq = "".join(
            itos[j]
            for i in range(len(bits) // vocab_size)
            for j in range(vocab_size)
            if bits[i * vocab_size + j] and itos[j] != "[nop]"
        )
        selfies_list.append(seq)

        try:
            sm = sf.decoder(seq)
            mol = Chem.MolFromSmiles(sm)
            smiles_list.append(sm if mol else None)
        except Exception:
            smiles_list.append(None)

    save_predictions_multi(df, selfies_list, smiles_list, save_dir, group_name)


def run_no_clustering_stage(stage2_dir, model_root, label):
    print(f"\n=== Stage 3A: {label} (No-Clustering) ===")

    for group in os.listdir(model_root):
        model_dir = os.path.join(model_root, group)
        if not os.path.isdir(model_dir):
            continue

        data_file = os.path.join(stage2_dir, f"{group}_predictions.pkl")
        if not os.path.exists(data_file):
            continue

        df = pd.read_pickle(data_file)
        save_dir = ensure_dir(os.path.join(STAGE3_NOCLUST_OUT, f"{label}_{group}"))

        try:
            predict_no_clustering(df, model_dir, save_dir, group)
        except Exception as e:
            print(f"❌ {group}: {e}")


# ==============================================================
# ============== STAGE 3B  (With Clustering) ====================
# ==============================================================

def run_clustering_stage(part_dir, stage2_dir, label):
    print(f"\n=== Stage 3B: {label} (With Clustering) ===")

    for group in os.listdir(part_dir):
        group_dir = os.path.join(part_dir, group)
        if not os.path.isdir(group_dir):
            continue

        pred_file = os.path.join(stage2_dir, f"{group}_predictions.pkl")
        if not os.path.exists(pred_file):
            print(f"⚠️ No Stage2 file for {group}, skipping.")
            continue

        vocab_path = os.path.join(group_dir, "Models", "vocab_info.pkl")
        kmeans_path = os.path.join(group_dir, "K_mean", "kmeans_model.pkl")
        models_dir = os.path.join(group_dir, "Models")

        if not (os.path.exists(vocab_path) and os.path.exists(kmeans_path)):
            print(f"⚠️ Missing clustering files for {group}.")
            continue

        with open(vocab_path, "rb") as f:
            vocab_info = pickle.load(f)

        symbol_to_idx = vocab_info["symbol_to_index"]  # token -> idx
        itos = {idx: sym for sym, idx in symbol_to_idx.items()}
        vocab_size = len(itos)

        with open(kmeans_path, "rb") as f:
            kmeans = pickle.load(f)

        df = pd.read_pickle(pred_file)
        mz_cols = [c for c in df.columns if str(c).startswith("mz")]
        X = df[mz_cols].values

        # Match KMeans feature count
        if hasattr(kmeans, "n_features_in_"):
            X = pad_or_truncate(X, int(kmeans.n_features_in_))

        # ✅ Robust dtype fix for sklearn KMeans: make BOTH X and centers float32 + contiguous
        X = np.ascontiguousarray(X, dtype=np.float32)
        if hasattr(kmeans, "cluster_centers_"):
            kmeans.cluster_centers_ = np.ascontiguousarray(kmeans.cluster_centers_, dtype=np.float32)

        clusters = kmeans.predict(X)

        group_save = ensure_dir(os.path.join(STAGE3_CLUST_OUT, f"{label}_{group}"))

        for cid in np.unique(clusters):
            mask = (clusters == cid)
            df_c = df.loc[mask].copy()
            Xc = X[mask]

            model_dir = os.path.join(models_dir, f"model{cid}")
            arch_file = os.path.join(model_dir, "architecture.json")
            if not os.path.exists(arch_file):
                print(f"⚠️ No model{cid} for {group}, skipping.")
                continue

            with open(arch_file) as f:
                arch = json.load(f)

            Xc = pad_or_truncate(Xc, int(arch["n_features"]))
            Xc = np.ascontiguousarray(Xc, dtype=np.float32)

            model = dc.models.MultitaskClassifier(
                n_tasks=arch["n_tasks"],
                n_features=arch["n_features"],
                layer_sizes=arch["layer_sizes"],
                dropouts=arch["dropout_rate"],
                model_dir=model_dir
            )
            model.restore()

            with torch.no_grad():
                outputs = model.model(torch.tensor(Xc, dtype=torch.float32))
                outputs = outputs[0] if isinstance(outputs, tuple) else outputs
                preds = outputs.cpu().numpy()

            selfies_list, smiles_list = [], []

            for pr in preds:
                if pr.ndim == 2 and pr.shape[-1] == 2:
                    pr = pr[:, 1]
                b = (pr >= 0.5).astype(int)

                usable = (len(b) // vocab_size) * vocab_size
                b = b[:usable]

                seq = "".join(
                    itos[j]
                    for i in range(len(b) // vocab_size)
                    for j in range(vocab_size)
                    if b[i * vocab_size + j] and itos[j] != "[nop]"
                )
                selfies_list.append(seq)

                try:
                    sm = sf.decoder(seq)
                    mol = Chem.MolFromSmiles(sm)
                    smiles_list.append(sm if mol else None)
                except Exception:
                    smiles_list.append(None)

            df_c["Cluster_ID"] = int(cid)
            cluster_dir = ensure_dir(os.path.join(group_save, f"cluster_{cid}"))

            save_predictions_multi(df_c, selfies_list, smiles_list, cluster_dir, f"{group}_cluster{cid}")


# ==============================================================
# ======================= MAIN PIPELINE =========================
# ==============================================================

if __name__ == "__main__":
    print("🚀 Starting Full Automatic Pipeline")

    stage1_dir = stage1_classify(DATA_FILE, STAGE1_MODEL, STAGE1_LABEL, BASE_DIR)

    stage2_A_dir = stage2_classify("A", stage1_dir, STAGE2_A_MODEL, STAGE2_A_LABEL)
    stage2_NA_dir = stage2_classify("NA", stage1_dir, STAGE2_NA_MODEL, STAGE2_NA_LABEL)

    # Stage 3A: No clustering (final outputs)
    if stage2_A_dir:
        run_no_clustering_stage(stage2_A_dir, LAST_STAGE_A_DIR, "Aromatic")
    if stage2_NA_dir:
        run_no_clustering_stage(stage2_NA_dir, LAST_STAGE_NA_DIR, "NonAromatic")

    # Stage 3B: With clustering (final outputs)
    aromatic_cluster_dir = os.path.join(CLUSTER_BASE_DIR, "Aromatic_part")
    non_aromatic_cluster_dir = os.path.join(CLUSTER_BASE_DIR, "Non_Aromatic_part")

    if os.path.exists(aromatic_cluster_dir) and stage2_A_dir:
        run_clustering_stage(aromatic_cluster_dir, stage2_A_dir, "Aromatic")

    if os.path.exists(non_aromatic_cluster_dir) and stage2_NA_dir:
        run_clustering_stage(non_aromatic_cluster_dir, stage2_NA_dir, "NonAromatic")

    print("\n🎯 ALL STAGES COMPLETED SUCCESSFULLY!")
